In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# Import necessary libraries
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

# Set the paths to your data
base_dir = '/kaggle/input/indian-birds/Birds_25'
train_dir = os.path.join(base_dir, 'train')
valid_dir = os.path.join(base_dir, 'valid')

# Load the training dataset and limit to 50% by taking only half
print("Loading and sampling 50% of the training dataset...")
train_dataset = image_dataset_from_directory(
    train_dir,
    image_size=(190, 190),
    batch_size=32,  # Using smaller batch size
    label_mode='int'
)

# Take 50% of the dataset (after shuffling)
train_dataset = train_dataset.take(len(train_dataset) // 1)

# Load the validation dataset and limit to 50% by taking only half
print("Loading and sampling 50% of the validation dataset...")
valid_dataset = image_dataset_from_directory(
    valid_dir,
    image_size=(190, 190),
    batch_size=32,  # Using smaller batch size
    label_mode='int'
)

# Take 50% of the validation dataset
valid_dataset = valid_dataset.take(len(valid_dataset) // 1)

# Normalize the datasets
def normalize_image(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

print("Normalizing datasets...")
train_dataset = train_dataset.map(normalize_image)
valid_dataset = valid_dataset.map(normalize_image)

# Cache and prefetch datasets for performance
train_dataset = train_dataset.cache().prefetch(buffer_size=tf.data.experimental.AUTOTUNE)
valid_dataset = valid_dataset.cache().prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

# Define the EfficientNetB0 model with pre-trained weights
print("Initializing EfficientNetB0 model with pre-trained weights...")
base_model = EfficientNetB0(
    include_top=False,                # Do not include the fully connected layers
    input_shape=(190, 190, 3),        # Set your input shape
    weights='imagenet'                # Load pre-trained weights
)

# Freeze the base model's layers to retain pre-trained features
base_model.trainable = False

# Add custom layers on top of the base model
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)  # Apply Global Average Pooling
x = layers.Dense(1024, activation='relu')(x)  # Fully connected layer
x = layers.Dropout(0.5)(x)  # Dropout to avoid overfitting
predictions = layers.Dense(25, activation='softmax')(x)  # Output layer for 25 classes

# Define the model
model = models.Model(inputs=base_model.input, outputs=predictions)

# Compile the model
model.compile(
    optimizer=Adam(),
    loss='sparse_categorical_crossentropy',  # using integer labels
    metrics=['accuracy']
)

# Summarize the model architecture
model.summary()

# Set the number of epochs
epochs = 35  # You can change this value based on your needs

# Add early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Add model checkpoint callback to save best model
checkpoint_callback = ModelCheckpoint(
    'best_model_pretrained_efficientnet.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',  # Save the best model based on validation loss
    verbose=1
)

# Train the model on 50% of the data
print(f"Starting training on 50% of the dataset for {epochs} epochs...")
history = model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=epochs,
    callbacks=[early_stopping, checkpoint_callback]
)

# Unfreeze the base model's layers for fine-tuning
print("Unfreezing base model for fine-tuning...")
base_model.trainable = True
model.compile(
    optimizer=Adam(learning_rate=1e-5),  # Use a lower learning rate for fine-tuning
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Fine-tune the model
print("Fine-tuning the model...")
history_fine = model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=epochs // 2,  # Fine-tune for fewer epochs
    callbacks=[early_stopping, checkpoint_callback]
)

# Evaluate the model
print("Evaluating the model on the validation dataset...")
loss, accuracy = model.evaluate(valid_dataset)
print(f"Validation Loss: {loss}")
print(f"Validation Accuracy: {accuracy}")

# Save the fine-tuned model
print("Saving the fine-tuned model...")
model.save('fine_tuned_pretrained_efficientnetb0.h5')


Loading and sampling 50% of the training dataset...
Found 30000 files belonging to 25 classes.
Loading and sampling 50% of the validation dataset...
Found 7500 files belonging to 25 classes.
Normalizing datasets...
Initializing EfficientNetB0 model with pre-trained weights...
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 190, 190,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 190, 190,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 190, 190,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 190, 190,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 191, 191,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 95, 95,    │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 95, 95,    │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 95, 95,    │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 95, 95,    │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 95, 95,    │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 95, 95,    │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 95, 95,    │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 95, 95,    │        512 │ block1a_se_excit

 Total params: 5,386,940 (20.55 MB)

 Trainable params: 1,337,369 (5.10 MB)

...

Starting training on 50% of the dataset for 35 epochs...
Epoch 1/35


I0000 00:00:1733530217.256593      93 service.cc:145] XLA service 0x79a8b0004940 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1733530217.256663      93 service.cc:153]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0


  5/938 ━━━━━━━━━━━━━━━━━━━━ 24s 26ms/step - accuracy: 0.0041 - loss: 3.2771     

I0000 00:00:1733530236.597101      93 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


938/938 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - accuracy: 0.0387 - loss: 3.2404
Epoch 1: val_loss improved from inf to 3.21894, saving model to best_model_pretrained_efficientnet.keras
938/938 ━━━━━━━━━━━━━━━━━━━━ 173s 154ms/step - accuracy: 0.0387 - loss: 3.2404 - val_accuracy: 0.0400 - val_loss: 3.2189
Epoch 2/35
937/938 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.0336 - loss: 3.2195
Epoch 2: val_loss improved from 3.21894 to 3.21893, saving model to best_model_pretrained_efficientnet.keras
938/938 ━━━━━━━━━━━━━━━━━━━━ 26s 28ms/step - accuracy: 0.0336 - loss: 3.2195 - val_accuracy: 0.0400 - val_loss: 3.2189
Epoch 3/35
938/938 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.0365 - loss: 3.2193
Epoch 3: val_loss improved from 3.21893 to 3.21893, saving model to best_model_pretrained_efficientnet.keras
938/938 ━━━━━━━━━━━━━━━━━━━━ 26s 28ms/step - accuracy: 0.0365 - loss: 3.2193 - val_accuracy: 0.0400 - val_loss: 3.2189
Epoch 4/35
936/938 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.

In [1]:
# Generate the classification report and confusion matrix
print("Generating classification report and confusion matrix...")
print("Classification Report:")
print(classification_report(y_true, y_pred))

print("Confusion Matrix:")
conf_matrix = confusion_matrix(y_true, y_pred)

# Plot the confusion matrix
plt.figure(figsize=(10, 8))
plt.imshow(conf_matrix, cmap='Blues')
plt.colorbar()
plt.title("Confusion Matrix")
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(np.arange(25), [f'Class {i}' for i in range(25)], rotation=90)
plt.yticks(np.arange(25), [f'Class {i}' for i in range(25)])
plt.show()


Generating classification report and confusion matrix...
Classification Report:


NameError: name 'classification_report' is not defined